# 04 Local Warm-start Laptop

Notebook ini versi lokal laptop untuk eksperimen warm-start. Sekarang sengaja pakai `configs/local_warmstart.yaml`, sama seperti `03`, supaya tuning CER/bootstrap v2 yang baru langsung kepakai.

Kalau mau training from-scratch `facebook/wav2vec2-xls-r-300m`, pakai `02_prepare_train_loop.ipynb` atau ganti config secara manual ke `configs/local_3050.yaml`.


In [55]:
from pathlib import Path
import os
import sys
import subprocess

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for path in candidates:
        if (path / "quran_asr").exists() and (path / "configs").exists() and (path / "scripts").exists():
            return path
    raise RuntimeError(
        "Root project tidak ketemu. Jalankan notebook dari folder repo atau dari folder notebooks di dalam repo.\n"
        f"Posisi sekarang: {current}"
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

CACHE_DIR = PROJECT_ROOT / ".cache" / "hf"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR / "transformers")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.executable)
print("HF_HOME:", os.environ["HF_HOME"])


PROJECT_ROOT: /home/jrilym/Projects/AI/model-asr-quran
Python: /home/jrilym/Projects/AI/model-asr-quran/.venv/bin/python
HF_HOME: /home/jrilym/Projects/AI/model-asr-quran/.cache/hf


In [56]:
import yaml

CONFIG_PATH = PROJECT_ROOT / "configs" / "local_warmstart.yaml"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config belum ada: {CONFIG_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("CONFIG_PATH:", CONFIG_PATH)
print("Run name:", cfg["run_name"])
print("Base model:", cfg["model"]["base"])
print("Processed dir:", PROJECT_ROOT / cfg["data"]["processed_dir"])
print("Output dir:", PROJECT_ROOT / cfg["logging"]["output_dir"])


CONFIG_PATH: /home/jrilym/Projects/AI/model-asr-quran/configs/local_warmstart.yaml
Run name: local_warmstart_cer_bootstrap_v2
Base model: rabah2026/wav2vec2-large-xlsr-53-arabic-quran-v_final
Processed dir: /home/jrilym/Projects/AI/model-asr-quran/data/processed
Output dir: /home/jrilym/Projects/AI/model-asr-quran/data/artifacts/checkpoints/local_warmstart_cer_bootstrap_v2


In [57]:
def run_cmd(args):
    args = [str(arg) for arg in args]
    print("Running:")
    print(" ".join(args))
    subprocess.run(args, check=True, cwd=PROJECT_ROOT)


In [58]:
run_cmd(["nvidia-smi"])

import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA tidak kebaca. Notebook ini disiapkan untuk laptop/PC NVIDIA GPU.")

print("GPU:", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print("VRAM GB:", round(props.total_memory / 1024**3, 2))


Running:
nvidia-smi
Wed Jul  8 22:37:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.43.02              KMD Version: 610.43.02     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...    Off |   00000000:01:00.0  On |                  N/A |
| N/A   43C    P8              7W /   60W |      53MiB /   6144MiB |     49%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [59]:
preflight = PROJECT_ROOT / "scripts" / "cloud_preflight.py"
if preflight.exists():
    run_cmd([sys.executable, preflight])
else:
    print("Preflight script tidak ada, skip:", preflight)


Running:
/home/jrilym/Projects/AI/model-asr-quran/.venv/bin/python /home/jrilym/Projects/AI/model-asr-quran/scripts/cloud_preflight.py

=== cloud pre-flight ===
  [OK  ] CUDA GPU — NVIDIA GeForce RTX 3050 6GB Laptop GPU
  [OK  ] ffmpeg — /usr/bin/ffmpeg
  [OK  ] dep accelerate
  [OK  ] dep torch
  [OK  ] dep torchaudio
  [OK  ] dep transformers
  [OK  ] dep datasets
  [OK  ] dep evaluate
  [OK  ] dep jiwer
  [OK  ] dep soundfile
  [OK  ] dep librosa
  [OK  ] dep huggingface_hub
  [OK  ] dep yaml
  [OK  ] dep requests
  [OK  ] dep tqdm
  [WARN] HF token — none — set logging.hub_repo=null or run notebook_login()
  [OK  ] disk free — 201.8 GB free on / (need ~15-20 GB)

0 hard failure(s). Fix FAIL items before training.


## 1. Cek Data Lokal

Notebook ini menganggap data sudah ada seperti workflow `02`. Kalau belum ada, jalankan cell download di bawah.


In [60]:
run_cmd([
    sys.executable,
    PROJECT_ROOT / "scripts" / "download.py",
    "--config",
    CONFIG_PATH,
])


Running:
/home/jrilym/Projects/AI/model-asr-quran/.venv/bin/python /home/jrilym/Projects/AI/model-asr-quran/scripts/download.py --config /home/jrilym/Projects/AI/model-asr-quran/configs/local_warmstart.yaml


[text] cache ready: data/raw/text/quran_uthmani.json
[audio] 6236 ayat × 1 reciter(s)
Husary_128kbps_Mujawwad: 100%|██████████| 6236/6236 [00:30<00:00, 206.82it/s]
[Husary_128kbps_Mujawwad] {'cached': 6236}


In [61]:
RAW_AUDIO_DIR = PROJECT_ROOT / "data" / "raw" / "audio"
RAW_TEXT_FILE = PROJECT_ROOT / "data" / "raw" / "text" / "quran_uthmani.json"

if not RAW_AUDIO_DIR.exists() or not any(RAW_AUDIO_DIR.iterdir()):
    raise RuntimeError(
        f"Audio belum ada di: {RAW_AUDIO_DIR}\n"
        "Jalankan cell download dulu kalau data belum tersedia."
    )

if not RAW_TEXT_FILE.exists():
    raise FileNotFoundError(
        f"Text Quran belum ada: {RAW_TEXT_FILE}\n"
        "Jalankan cell download dulu kalau data belum tersedia."
    )

audio_files = list(RAW_AUDIO_DIR.rglob("*.*"))
print("Raw audio OK:", RAW_AUDIO_DIR)
print("Jumlah audio:", len(audio_files))
print("Quran text OK:", RAW_TEXT_FILE)
print("\nContoh audio:")
for file in audio_files[:5]:
    print("-", file.relative_to(PROJECT_ROOT))


Raw audio OK: /home/jrilym/Projects/AI/model-asr-quran/data/raw/audio
Jumlah audio: 24946
Quran text OK: /home/jrilym/Projects/AI/model-asr-quran/data/raw/text/quran_uthmani.json

Contoh audio:
- data/raw/audio/Husary_128kbps_Mujawwad/001003.md5
- data/raw/audio/Husary_128kbps_Mujawwad/001002.md5
- data/raw/audio/Husary_128kbps_Mujawwad/001001.md5
- data/raw/audio/Husary_128kbps_Mujawwad/001004.md5
- data/raw/audio/Husary_128kbps_Mujawwad/001005.md5


## 2. Build Dataset dan Vocab


In [62]:
run_cmd([
    sys.executable,
    PROJECT_ROOT / "scripts" / "build.py",
    "--config",
    CONFIG_PATH,
])


Running:
/home/jrilym/Projects/AI/model-asr-quran/.venv/bin/python /home/jrilym/Projects/AI/model-asr-quran/scripts/build.py --config /home/jrilym/Projects/AI/model-asr-quran/configs/local_warmstart.yaml


Saving the dataset (1/1 shards): 100%|██████████| 199/199 [00:00<00:00, 72121.88 examples/s]
  train: 1611 rows
  validation: 165 rows
  test: 199 rows
[build] saved DatasetDict to data/processed


In [63]:
run_cmd([
    sys.executable,
    PROJECT_ROOT / "scripts" / "build_vocab.py",
    "--config",
    CONFIG_PATH,
])


Running:
/home/jrilym/Projects/AI/model-asr-quran/.venv/bin/python /home/jrilym/Projects/AI/model-asr-quran/scripts/build_vocab.py --config /home/jrilym/Projects/AI/model-asr-quran/configs/local_warmstart.yaml
{
 "َ": 0,
 "ِ": 1,
 "ْ": 2,
 "ل": 3,
 "ُ": 4,
 "ن": 5,
 "م": 6,
 "ا": 7,
 "و": 8,
 "ّ": 9,
 "ي": 10,
 "ر": 11,
 "ٱ": 12,
 "ه": 13,
 "ب": 14,
 "ك": 15,
 "ٰ": 16,
 "ت": 17,
 "ع": 18,
 "ف": 19,
 "أ": 20,
 "ق": 21,
 "ى": 22,
 "س": 23,
 "د": 24,
 "إ": 25,
 "ذ": 26,
 "ح": 27,
 "ٓ": 28,
 "ٍ": 29,
 "ج": 30,
 "ً": 31,
 "ٌ": 32,
 "ص": 33,
 "خ": 34,
 "ة": 35,
 "ء": 36,
 "ز": 37,
 "ش": 38,
 "ط": 39,
 "ث": 40,
 "غ": 41,
 "ض": 42,
 "ئ": 43,
 "ظ": 44,
 "ٔ": 45,
 "ؤ": 46,
 "|": 47,
 "[UNK]": 48,
 "[PAD]": 49
}


[vocab] 50 tokens from dataset(train+validation) (1776 texts) -> data/artifacts/vocab.json


In [64]:
PROCESSED_DIR = PROJECT_ROOT / cfg["data"]["processed_dir"]
VOCAB_PATH = PROJECT_ROOT / cfg["model"]["vocab_path"]
OUTPUT_DIR = PROJECT_ROOT / cfg["logging"]["output_dir"]

print("Processed:", PROCESSED_DIR, "OK" if PROCESSED_DIR.exists() else "MISSING")
print("Vocab:", VOCAB_PATH, "OK" if VOCAB_PATH.exists() else "MISSING")
print("Output:", OUTPUT_DIR)

if PROCESSED_DIR.exists():
    print("\nIsi data/processed:")
    for path in sorted(PROCESSED_DIR.rglob("*"))[:50]:
        print("-", path.relative_to(PROJECT_ROOT))


Processed: /home/jrilym/Projects/AI/model-asr-quran/data/processed OK
Vocab: /home/jrilym/Projects/AI/model-asr-quran/data/artifacts/vocab.json OK
Output: /home/jrilym/Projects/AI/model-asr-quran/data/artifacts/checkpoints/local_warmstart_cer_bootstrap_v2

Isi data/processed:
- data/processed/dataset_dict.json
- data/processed/test
- data/processed/test/cache-2104c9c23f1319f3.arrow
- data/processed/test/cache-9d88e28de32a507d.arrow
- data/processed/test/cache-a13591b069964ba2.arrow
- data/processed/test/cache-c88ec2bdbd80a326.arrow
- data/processed/test/cache-d5941853398bf246.arrow
- data/processed/test/data-00000-of-00001.arrow
- data/processed/test/dataset_info.json
- data/processed/test/state.json
- data/processed/train
- data/processed/train/cache-07bd9501f34472b0.arrow
- data/processed/train/cache-4fcd92e88f7864e2.arrow
- data/processed/train/cache-bf257d5982d0baa5.arrow
- data/processed/train/cache-ee3a8a57b79a2086.arrow
- data/processed/train/cache-efecaaca69f9cb90.arrow
- data/

## 3. Training Lokal

Mengikuti recipe `02`: pakai `train_manual.py` supaya staging bootstrap -> finetune jalan.


In [65]:
import yaml

CONFIG_PATH = PROJECT_ROOT / "configs" / "local_warmstart.yaml"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config belum ada: {CONFIG_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("CONFIG_PATH:", CONFIG_PATH)
print("Run name:", cfg["run_name"])
print("Output dir:", PROJECT_ROOT / cfg["logging"]["output_dir"])

stage_keys = [
    "auto_stage",
    "bootstrap_min_epochs",
    "bootstrap_max_epochs",
    "bootstrap_head_learning_rate",
    "bootstrap_blank_logit_bias_init",
    "finetune_encoder_trainable_layers",
    "finetune_encoder_learning_rate",
    "finetune_head_learning_rate",
]

print("Manual trainer recipe:")
for key in stage_keys:
    print(f"{key}:", cfg["training"].get(key))

run_cmd([
    sys.executable,
    PROJECT_ROOT / "scripts" / "train_manual.py",
    "--config",
    CONFIG_PATH,
])


CONFIG_PATH: /home/jrilym/Projects/AI/model-asr-quran/configs/local_warmstart.yaml
Run name: local_warmstart_cer_bootstrap_v2
Output dir: /home/jrilym/Projects/AI/model-asr-quran/data/artifacts/checkpoints/local_warmstart_cer_bootstrap_v2
Manual trainer recipe:
auto_stage: True
bootstrap_min_epochs: 5
bootstrap_max_epochs: 8
bootstrap_head_learning_rate: 0.001
bootstrap_blank_logit_bias_init: -2.0
finetune_encoder_trainable_layers: 1
finetune_encoder_learning_rate: 2e-07
finetune_head_learning_rate: 0.0002
Running:
/home/jrilym/Projects/AI/model-asr-quran/.venv/bin/python /home/jrilym/Projects/AI/model-asr-quran/scripts/train_manual.py --config /home/jrilym/Projects/AI/model-asr-quran/configs/local_warmstart.yaml


Loading weights: 100%|██████████| 424/424 [00:00<00:00, 957.10it/s]
[transformers] Wav2Vec2ForCTC LOAD REPORT from: rabah2026/wav2vec2-large-xlsr-53-arabic-quran-v_final
Key            | Status   |                                                                                           
---------------+----------+-------------------------------------------------------------------------------------------
lm_head.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([56, 1024]) vs model:torch.Size([52, 1024])
lm_head.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([56]) vs model:torch.Size([52])            

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Initial blank logit bias adjusted by -2.00
No resume checkpoint found. Starting fresh.
Model: rabah2026/wav2vec2-large-xlsr-53-arabic-quran-v_final
Trainable encoder layers: 0
Trainable params: 53300
Total params: 315492020
Trainable ratio: 0.0169 %
epoch     stage   step     train      eval    WERp    CERp    WERd    CERd   empty  best


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    1 bootstrap    101   24.8142    4.1924   0.997   0.825   1.003   0.764    3.6%     * cer_plain
[WARN] empty/harakat-only decode rate: 3.6%


...

    3 bootstrap    303   10.4232    3.7048   1.000   0.858   1.005   0.782    9.1%     - cer_plain
[WARN] empty/harakat-only decode rate: 9.1%
No improvement for 2 epoch(s).


Epoch 4/15:   0%|          | 0/1611 [00:00<?, ?it/s]

Saved training state to: data/artifacts/checkpoints/local_warmstart_cer_bootstrap_v2/latest


Epoch 4/15:   9%|▊         | 138/1611 [00:37<06:36,  3.72it/s, loss=6.47, step=311]
Traceback (most recent call last):
  File "/home/jrilym/Projects/AI/model-asr-quran/scripts/train_manual.py", line 37, in <module>
    raise SystemExit(main())
                     ^^^^^^
  File "/home/jrilym/Projects/AI/model-asr-quran/scripts/train_manual.py", line 24, in main
    result = train_manual(
             ^^^^^^^^^^^^^
  File "/home/jrilym/Projects/AI/model-asr-quran/quran_asr/training/manual_train.py", line 158, in train_manual
    train_loss, global_step = _train_one_epoch(
                              ^^^^^^^^^^^^^^^^^
  File "/home/jrilym/Projects/AI/model-asr-quran/quran_asr/training/manual_train.py", line 413, in _train_one_epoch
    for step, batch in enumerate(progress):
                       ^^^^^^^^^^^^^^^^^^^
  File "/home/jrilym/Projects/AI/model-asr-quran/.venv/lib/python3.12/site-packages/tqdm/std.py", line 1181, in __iter__
    for obj in iterable:
               ^^^^^^^^
 

KeyboardInterrupt: 

## 4. Cek Hasil


In [66]:
FINAL = PROJECT_ROOT / cfg["logging"]["output_dir"] / "best"
AUDIO = PROJECT_ROOT / "data" / "raw" / "audio" / "Husary_128kbps_Mujawwad" / "001001.wav"

if not FINAL.exists():
    raise FileNotFoundError(f"Checkpoint final belum ada: {FINAL}")
if not AUDIO.exists():
    raise FileNotFoundError(f"Audio sanity check belum ada: {AUDIO}")

run_cmd([
    sys.executable,
    PROJECT_ROOT / "scripts" / "correct.py",
    "--model-dir",
    FINAL,
    "--audio",
    AUDIO,
    "--text",
    "بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ",
])


Running:
/home/jrilym/Projects/AI/model-asr-quran/.venv/bin/python /home/jrilym/Projects/AI/model-asr-quran/scripts/correct.py --model-dir /home/jrilym/Projects/AI/model-asr-quran/data/artifacts/checkpoints/local_warmstart_cer_bootstrap_v2/best --audio /home/jrilym/Projects/AI/model-asr-quran/data/raw/audio/Husary_128kbps_Mujawwad/001001.wav --text بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ


Loading weights: 100%|██████████| 424/424 [00:00<00:00, 847.68it/s]


  #  status        conf   start-end     word
  0  skipped       0.00    0.00-0.12    بِسْمِ
  1  skipped       0.00    0.14-4.84    ٱللَّهِ
  2  skipped       0.02    4.86-6.46    ٱلرَّحْمَٰنِ
  3  skipped       0.01    6.48-10.40   ٱلرَّحِيمِ


In [67]:
import json
import pandas as pd

OUTPUT_DIR = PROJECT_ROOT / cfg["logging"]["output_dir"]
history_path = OUTPUT_DIR / "training_history.json"
latest_history_path = OUTPUT_DIR / "latest" / "training_history.json"

if latest_history_path.exists():
    history_path = latest_history_path

if not history_path.exists():
    raise FileNotFoundError(f"History belum ada: {history_path}")

history = json.loads(history_path.read_text(encoding="utf-8"))
df_history = pd.DataFrame(history)
display(df_history.tail(20))

csv_path = OUTPUT_DIR / "latest" / "training_history.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)
df_history.to_csv(csv_path, index=False)
print("CSV saved to:", csv_path)


,epoch,stage,global_step,train_loss,eval_loss,wer,cer,wer_plain,cer_plain,empty_pred_rate
0,1,bootstrap,101,24.814185,4.192416,1.003257,0.763962,0.996743,0.824808,0.036364
1,2,bootstrap,202,11.845979,3.848999,1.013029,0.770943,1.001629,0.837513,0.036364
2,3,bootstrap,303,10.423212,3.704792,1.004886,0.782453,1.000000,0.857907,0.090909


CSV saved to: /home/jrilym/Projects/AI/model-asr-quran/data/artifacts/checkpoints/local_warmstart_cer_bootstrap_v2/latest/training_history.csv
